In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import getpass, os
from huggingface_hub import login

hf_token = getpass.getpass("Paste Hugging Face token (hidden): ")

login(token=hf_token, add_to_git_credential=False)
del hf_token

Paste Hugging Face token (hidden): ··········


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# Memory optimizations
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception as e:
    print("xFormers not enabled:", e)
pipe.enable_attention_slicing()

print("Pipeline loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

text_encoder/model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Pipeline loaded.


In [ ]:
import os, random, json, math, time
from dataclasses import dataclass
from typing import Dict, List

CATEGORIES = ["apple", "tomato", "orange", "banana", "cucumber"]
N_PER_CLASS = 120
OUT_DIR = "/content/drive/MyDrive/fruitveg_data3"
IMG_DIR = f"{OUT_DIR}/images"
META_JSONL = f"{OUT_DIR}/metadata3.jsonl"

os.makedirs(IMG_DIR, exist_ok=True)

SEASONS = ["spring", "summer", "autumn", "winter"]
ORIGINS = ["Türkiye", "Spain", "USA", "Mexico", "Italy", "Morocco", "Brazil", "Egypt"]
RIPENESS = ["under-ripe", "ripe", "over-ripe"]

WEIGHT_RANGES = {
    "apple":    (120, 250),
    "banana":   (90, 180),
    "orange":   (150, 300),
    "tomato":   (70, 200),
    "cucumber": (150, 400),
}

COLORS = {
    "apple":    ["red", "green"],
    "banana":   ["yellow", "green"],
    "orange":   ["orange"],
    "tomato":   ["red", "yellow", "green"],
    "cucumber": ["green"],
}

SCENES = [
    "studio product photo on neutral background",
    "on a wooden cutting board, soft daylight",
    "in a market basket, natural light",
    "top-down view on marble background",
]


GUIDANCE = 6.5
STEPS = 24

H, W = 512, 512

print("Config OK")


Config OK


In [ ]:
import numpy as np

def clip_gauss_int(low, high, mu=None, sigma=None):
    if mu is None: mu = 0.5*(low+high)
    if sigma is None: sigma = (high-low)/6
    v = int(np.random.normal(mu, sigma))
    return max(low, min(high, v))

def sample_attributes(klass: str):
    color = random.choice(COLORS[klass])
    season = random.choice(SEASONS)
    origin = random.choice(ORIGINS)
    wmin, wmax = WEIGHT_RANGES[klass]
    weight_g = clip_gauss_int(wmin, wmax)
    ripeness = random.choices(RIPENESS, weights=[1, 4, 1])[0]
    defects_flag = random.random() < 0.15
    scene = random.choice(SCENES)
    style = random.choice([
        "high detail, natural colors",
        "soft light, product photography",
        "diffused light, editorial style",
        "studio lighting, crisp focus",
    ])
    defect_txt = ", slight blemishes" if defects_flag else ", clean surface"
    prompt = f"{scene}, {style}, a {color} {klass}{defect_txt}"
    return {
        "class": klass,
        "color": color,
        "season": season,
        "origin": origin,
        "weight_g": weight_g,
        "ripeness": ripeness,
        "defects_flag": defects_flag,
        "prompt": prompt,
    }


In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 3
random_seed_base = 1337

STEPS = 30
GUIDANCE = 7.5
H = 512
W = 512

CLASS_COUNTS = {
      "apple" : None,
      "banana" : None,
      "cucumber" : None,
      "orange" : None,
      "tomato" : None
}

def make_filename(gid, klass):
    return f"{klass}_set3_{gid:06d}.png"


global_id = 0
to_generate = []

for klass, target in CLASS_COUNTS.items():
    for _ in range(target):
        to_generate.append((global_id, klass))
        global_id += 1

print(f"Planned {len(to_generate)} images total.")

meta_f = open(META_JSONL, "a", encoding="utf-8")

pbar = tqdm(total=len(to_generate))
generated = 0

idx = 0
while idx < len(to_generate):

    batch_prompts = []
    batch_rows = []
    batch_paths = []
    seeds = []

    while len(batch_prompts) < BATCH_SIZE and idx < len(to_generate):

        gid, klass = to_generate[idx]
        idx += 1

        fname = make_filename(gid, klass)
        img_path = f"{IMG_DIR}/{fname}"

        if os.path.exists(img_path):
            pbar.update(1)
            continue

        row = sample_attributes(klass)
        seed = random_seed_base + gid

        seeds.append(seed)
        batch_prompts.append(row["prompt"])
        batch_rows.append((gid, klass, row))
        batch_paths.append(img_path)

    if not batch_prompts:
        continue

    gens = [torch.Generator(device="cuda").manual_seed(s) for s in seeds]

    images = pipe(
        prompt=batch_prompts,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        height=H,
        width=W,
        generator=gens,
    ).images

    for (gid, klass, row), img, seed, path in zip(batch_rows, images, seeds, batch_paths):

        img.save(path)

        record = {
            "id": gid,
            "class": klass,
            "image_path": path,
            **row,
            "seed": seed,
            "height": H,
            "width": W,
            "steps": STEPS,
            "guidance": GUIDANCE,
            "model": model_id,
        }

        meta_f.write(json.dumps(record, ensure_ascii=False) + "\n")

        generated += 1
        pbar.update(1)



meta_f.close()
print(f"Generated now: {generated} images. Metadata at {META_JSONL}")


Planned 50 images total.


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Generated now: 50 images. Metadata at /content/drive/MyDrive/fruitveg_data3/metadata3.jsonl
